# 01 - Exploratory Data Analysis (EDA)

## AI-Generated Text Detection Project

Dataset: DAIGT V2 Train Dataset

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/raw/train_v2_drcat_02.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Data Overview

In [ ]:
print('Dataset Info:')
print(df.info())
print('\n' + '='*50 + '\n')
print('Missing Values:')
print(df.isnull().sum())
print('\n' + '='*50 + '\n')
print('Duplicates:')
print(f'Total duplicates: {df.duplicated().sum()}')

## 3. Class Distribution

In [ ]:
print('Class Distribution:')
print(df['label'].value_counts())
print('\nClass Proportion:')
print(df['label'].value_counts(normalize=True))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

df['label'].value_counts().plot(kind='bar', ax=ax[0], color=['steelblue', 'coral'])
ax[0].set_title('Class Distribution (Count)')
ax[0].set_xlabel('Label (0=Human, 1=AI)')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['Human (0)', 'AI (1)'], rotation=0)

df['label'].value_counts().plot(kind='pie', ax=ax[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
ax[1].set_title('Class Distribution (Proportion)')
ax[1].set_ylabel('')
ax[1].legend(['Human (0)', 'AI (1)'])

plt.tight_layout()
plt.savefig('../app/assets/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Text Length Analysis

In [ ]:
df['text_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
df['sentence_count'] = df['text'].apply(lambda x: len(str(x).split('.')))

In [ ]:
print('Text Length Statistics by Class:')
print(df.groupby('label')[['text_length', 'word_count', 'sentence_count']].describe())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=df, x='text_length', hue='label', kde=True, ax=ax[0], palette=['steelblue', 'coral'])
ax[0].set_title('Distribution of Text Length (Characters)')
ax[0].legend(['AI (1)', 'Human (0)'])

sns.histplot(data=df, x='word_count', hue='label', kde=True, ax=ax[1], palette=['steelblue', 'coral'])
ax[1].set_title('Distribution of Word Count')
ax[1].legend(['AI (1)', 'Human (0)'])

sns.boxplot(data=df, x='label', y='word_count', ax=ax[2], palette=['steelblue', 'coral'])
ax[2].set_title('Word Count by Class')
ax[2].set_xlabel('Label (0=Human, 1=AI)')

plt.tight_layout()
plt.savefig('../app/assets/text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Source Distribution

In [ ]:
if 'source' in df.columns:
    print('Source Distribution:')
    print(df['source'].value_counts())
    
    fig, ax = plt.subplots(figsize=(12, 6))
    source_label = df.groupby(['source', 'label']).size().unstack(fill_value=0)
    source_label.plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
    ax.set_title('Source Distribution by Class')
    ax.set_xlabel('Source')
    ax.set_ylabel('Count')
    ax.legend(['Human (0)', 'AI (1)'])
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../app/assets/source_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Word Cloud

In [ ]:
human_text = ' '.join(df[df['label']==0]['text'].dropna().astype(str))
ai_text = ' '.join(df[df['label']==1]['text'].dropna().astype(str))

fig, ax = plt.subplots(1, 2, figsize=(20, 8))

wordcloud_human = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(human_text)
ax[0].imshow(wordcloud_human, interpolation='bilinear')
ax[0].set_title('Word Cloud - Human Written', fontsize=16)
ax[0].axis('off')

wordcloud_ai = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(ai_text)
ax[1].imshow(wordcloud_ai, interpolation='bilinear')
ax[1].set_title('Word Cloud - AI Generated', fontsize=16)
ax[1].axis('off')

plt.tight_layout()
plt.savefig('../app/assets/wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. N-gram Analysis

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_ngrams(corpus, n=None, top_k=20):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:top_k]

In [ ]:
human_corpus = df[df['label']==0]['text'].dropna().astype(str)
ai_corpus = df[df['label']==1]['text'].dropna().astype(str)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (corpus, label_name) in enumerate([(human_corpus, 'Human'), (ai_corpus, 'AI')]):
    for jdx, n in enumerate([1, 2]):
        top_ngrams = get_top_ngrams(corpus, n=n, top_k=15)
        df_ngrams = pd.DataFrame(top_ngrams, columns=['ngram', 'count'])
        
        ax = axes[idx, jdx]
        sns.barplot(data=df_ngrams, x='count', y='ngram', ax=ax, palette='viridis')
        ax.set_title(f'Top 15 {n}-grams - {label_name}')
        ax.set_xlabel('Frequency')
        ax.set_ylabel('')

plt.tight_layout()
plt.savefig('../app/assets/ngram_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Train-Validation-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['label']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

print(f'Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')

In [ ]:
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
val_df = pd.DataFrame({'text': X_val, 'label': y_val})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

print('Train/Val/Test splits saved to data/processed/')

## 9. Summary

In [ ]:
print('='*60)
print('EDA SUMMARY')
print('='*60)
print(f'Total samples: {len(df)}')
print(f'Human samples: {df[df["label"]==0].shape[0]} ({df[df["label"]==0].shape[0]/len(df)*100:.1f}%)')
print(f'AI samples: {df[df["label"]==1].shape[0]} ({df[df["label"]==1].shape[0]/len(df)*100:.1f}%)')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Average word count - Human: {df[df["label"]==0]["word_count"].mean():.1f}')
print(f'Average word count - AI: {df[df["label"]==1]["word_count"].mean():.1f}')
print('='*60)